# 11 · Tüm Veri Setini Büyütülmüş Görüntüden Yeniden OCR'lama

**Neden:** RVL-CDIP görüntüleri en uzun kenarı 1000 piksel olacak şekilde küçültülmüş; kelime yüksekliği 7–12 piksel. Tesseract en iyi 300 dpi civarında, yani daha büyük harflerde çalışır. 21 Ekim denemesinde görüntüyü 2 kat büyütmek bazı belgelerde güvenli okunan kelime sayısını katlarca artırdı (ör. 35 → 718).

**Önceki pilot neden bunu görmedi:** 03'teki pilot deneyde büyütme, medyan filtre ve ikiliğe çevirmeyle **birlikte** denendi. 08'de gördüğümüz gibi medyan filtre yazıyı bozuyor, bu yüzden ham görüntü seçilmişti. Büyütme tek başına hiç denenmemişti.

**Bu notebook genel bir iyileştirmedir:** her belgeye aynı işlem (2 kat bikübik büyütme → aynı Tesseract ayarları) uygulanır, belgeye özel ayar yoktur.

| Bölüm | Ne yapılır |
|---|---|
| A | 4 sınıfın tüm belgeleri (train + validation + test) 2 kat büyütülüp yeniden OCR'lanır → `data/ocr_dataset_x2.csv` |
| B | OCR kalitesi karşılaştırılır: eski (ham) ve yeni (x2), sınıf bazında |
| C | Aynı modeller aynı ayarlarla yeni metinle yeniden eğitilir; hangi OCR'ın kullanılacağına **validation** setinde karar verilir, test sonucu en sonda raporlanır |

**Süre:** A bölümü yaklaşık 1,5–3 saat (Colab CPU). Yarıda kalırsa tekrar çalıştırınca kaldığı yerden devam eder. B ve C birkaç dakika.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract opencv-python-headless symspellpy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, os, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA, MODELS = ROOT / "data", ROOT / "models"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
CFG_FILE = ROOT / "siniflar.json"
CLASSES = json.loads(CFG_FILE.read_text())["CLASSES"] if CFG_FILE.exists() else ["email", "form", "invoice", "resume"]
TR = {"email": "E-posta", "invoice": "Fatura", "resume": "CV", "scientific_publication": "Bilimsel makale", "form": "Form"}
SEED = 42
print("Sınıflar:", CLASSES, "· CPU:", os.cpu_count())

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_core.py
import cv2
import numpy as np
import pytesseract


def estimate_skew(gray, max_angle=5.0, step=0.25):
    """Projection-profile skew estimate: the angle whose row sums are sharpest."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = cv2.medianBlur(th, 3)  # drop isolated noise pixels
    h, w = th.shape
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + 1e-9, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        r = cv2.warpAffine(th, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(r.sum(axis=1, dtype=np.float64)))
        if score > best_score:
            best_angle, best_score = float(a), score
    return best_angle


def deskew(gray):
    """Rotate the page so text lines are horizontal."""
    angle = estimate_skew(gray)
    if abs(angle) < 0.3:
        return gray, 0.0
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def preprocess(img_bgr_or_gray, upscale_to=2000, do_deskew=True, binarize=True):
    """Grayscale -> upscale (RVL-CDIP is max 1000px) -> denoise -> deskew -> Otsu."""
    img = img_bgr_or_gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    gray = cv2.medianBlur(gray, 3)  # salt-and-pepper scan noise, before upscaling
    h, w = gray.shape
    scale = upscale_to / max(h, w)
    if scale > 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    angle = 0.0
    if do_deskew:
        gray, angle = deskew(gray)
    if binarize:
        _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, angle


def run_ocr(img, lang="eng", psm=3, min_conf=0):
    """Tesseract OCR -> words, normalized boxes (0-1000, LayoutLM format), line text."""
    h, w = img.shape[:2]
    d = pytesseract.image_to_data(img, lang=lang, config=f"--oem 1 --psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    words, boxes, confs, lines = [], [], [], {}
    for i, txt in enumerate(d["text"]):
        txt = txt.strip()
        conf = float(d["conf"][i])
        if not txt or conf < min_conf:
            continue
        x, y, bw, bh = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        words.append(txt)
        confs.append(conf)
        boxes.append([int(1000 * x / w), int(1000 * y / h),
                      int(1000 * (x + bw) / w), int(1000 * (y + bh) / h)])
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(txt)
    text = "\n".join(" ".join(v) for _, v in sorted(lines.items()))
    return {
        "text": text,
        "words": words,
        "boxes": boxes,
        "confs": confs,
        "n_words": len(words),
        "mean_conf": float(np.mean(confs)) if confs else 0.0,
    }


def detect_table(gray):
    """Morfolojik işlemlerle yatay/dikey çizgileri bulup belgede tablo olup olmadığını tahmin eder.

    OCR tabloları bozarak okuduğu için (danışman/asistan görüşmesi) tablo bilgisi
    görüntüden ayrıca çıkarılır ve sınıflandırmada ek özellik olarak kullanılabilir.
    """
    img = gray if gray.ndim == 2 else cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    h, w = img.shape
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(10, w // 25), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(10, h // 25)))
    horiz = cv2.morphologyEx(th, cv2.MORPH_OPEN, hk)
    vert = cv2.morphologyEx(th, cv2.MORPH_OPEN, vk)
    n_h = len(cv2.findContours(horiz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    n_v = len(cv2.findContours(vert, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    inter = cv2.bitwise_and(cv2.dilate(horiz, np.ones((3, 3), np.uint8)), cv2.dilate(vert, np.ones((3, 3), np.uint8)))
    n_x = len(cv2.findContours(inter, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    line_ratio = float((horiz > 0).mean() + (vert > 0).mean())
    return {
        "n_hlines": n_h,
        "n_vlines": n_v,
        "n_intersections": n_x,
        "line_ratio": round(line_ratio, 5),
        "has_table": bool(n_h >= 3 and n_v >= 2 and n_x >= 4),
    }


def rotate_bound(gray, angle):
    """Görüntüyü kırpmadan döndürür (köşeler beyazla doldurulur)."""
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(gray, M, (nw, nh), flags=cv2.INTER_CUBIC, borderValue=255)


def _good_words(o, min_conf=70):
    return sum(c >= min_conf for c in o["confs"])


def upscale_if_small(gray, ocr, target_px=24):
    """Yazı küçükse görüntüyü büyütüp yeniden okur (21 Ekim).

    Tesseract harf yüksekliği ~20-30 piksel olduğunda en iyi çalışır. RVL-CDIP taramalarında kelime yüksekliği
    7-12 piksel; internetten alınan küçük resimlerde daha da az. Kelime kutularının ortanca yüksekliği
    hedefin altındaysa görüntü büyütülür; güvenli (>= 70) okunan kelime sayısı artarsa büyütülmüş okuma seçilir.
    """
    hs = [b[3] - b[1] for b in ocr["boxes"]]
    if not hs:
        med = 0.0
    else:
        med = float(np.median(hs)) * gray.shape[0] / 1000.0
    if med >= target_px * 0.8 and _good_words(ocr) > 0:
        return gray, ocr, 1.0
    f = 2.0   # denemelerde x2 en dengelisi: x3 bazı tablo sütunlarını kaybettirdi
    big = cv2.resize(gray, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    o2 = run_ocr(big)
    return (big, o2, f) if _good_words(o2) > 1.1 * _good_words(ocr) else (gray, ocr, 1.0)


def ocr_with_orientation(img, upscale=True):
    """Yan, eğik ya da ters dönmüş sayfayı düzeltip OCR yapar.

    21 Ekim: elle değerlendirmede ~77° dönük bir fatura hiç okunamamıştı ("WoRsN"). Yazı satırlarının açısı,
    satır toplamlarının en keskin olduğu açı aranarak (-90°…+90°) bulunur; satır yönü 180° belirsiz olduğu için
    sayfa o açıyla ve ters yönüyle okunur. Açı küçükse Tesseract'ın yön tespiti (OSD) baş aşağı sayfaya bakar.
    Aday hâller arasından en çok ve en emin kelimeyi veren seçilir; asıl hâlden en az %20 iyi değilse sayfaya
    dokunulmaz, böylece düz belgeler bozulmaz.
    Dönüş: (görüntü, ocr sonucu, döndürme açısı)
    """
    import re
    gray = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ocr = run_ocr(gray)
    score = lambda o: o["n_words"] * o["mean_conf"]
    angles = []
    a = estimate_skew(gray, max_angle=90, step=1.0)
    if abs(a) >= 3:
        angles += [a, a + 180]
    else:
        try:
            osd = pytesseract.image_to_osd(gray, config="--psm 0 -c min_characters_to_try=5")
            if int(re.search(r"Rotate: (\d+)", osd).group(1)) == 180:
                angles.append(180)
        except Exception:
            pass
    best = (gray, ocr, 0)
    for ang in angles:
        r = rotate_bound(gray, ang)
        o = run_ocr(r)
        if score(o) > 1.2 * score(best[1]):
            best = (r, o, ang)
    if upscale:
        g2, o2, _ = upscale_if_small(best[0], best[1])
        best = (g2, o2, best[2])
    return best


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_batch.py
"""Tüm veri setini aynı ayarlarla yeniden OCR'lar (21 Ekim: x2 büyütme deneyi).

Her belgeye aynı işlem uygulanır: gri görüntü → `scale` kat büyütme (bikübik) → Tesseract (LSTM, psm 3).
Belgeye özel hiçbir ayar yoktur. Sonuçlar parça parça CSV'ye yazılır; yarıda kalırsa kaldığı yerden devam eder.
"""
import os

os.environ.setdefault("OMP_THREAD_LIMIT", "1")   # her süreç tek çekirdek: paralel süreçler birbirini yavaşlatmasın

import time
from multiprocessing import Pool
from pathlib import Path

import cv2
import pandas as pd

from ocr_core import run_ocr


def _work(args):
    data_dir, rel_path, scale = args
    img = cv2.imread(str(Path(data_dir) / rel_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return {"path": rel_path, "text": "", "mean_conf": 0.0, "n_words": 0, "hata": "okunamadı"}
    if scale != 1:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    try:
        o = run_ocr(img)
        return {"path": rel_path, "text": o["text"], "mean_conf": round(o["mean_conf"], 2),
                "n_words": o["n_words"], "hata": ""}
    except Exception as e:  # tek belge hatası tüm işi durdurmasın
        return {"path": rel_path, "text": "", "mean_conf": 0.0, "n_words": 0, "hata": str(e)[:200]}


def run_batch(paths, data_dir, out_csv, scale=2.0, workers=2, chunk=40, log=print):
    out_csv = Path(out_csv)
    done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame(columns=["path"])
    todo = [p for p in paths if p not in set(done["path"])]
    log(f"Toplam {len(paths)} · daha önce yapılan {len(paths) - len(todo)} · kalan {len(todo)}")
    t0 = time.time()
    with Pool(workers) as pool:
        for k in range(0, len(todo), chunk):
            part = todo[k:k + chunk]
            rows = pool.map(_work, [(str(data_dir), p, scale) for p in part])
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
            tmp = out_csv.with_suffix(".tmp.csv")
            done.to_csv(tmp, index=False)
            tmp.replace(out_csv)                      # yarım yazılmış dosya kalmasın
            n = k + len(part)
            el = time.time() - t0
            log(f"{n}/{len(todo)} · {el / 60:.1f} dk · kalan ~{el / n * (len(todo) - n) / 60:.0f} dk")
    return done


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/text_clean.py
import re
import unicodedata

# OCR'ın sık ürettiği tipografik karakterleri sadeleştir
_CHAR_MAP = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "•": "-", "·": "-",
    " ": " ", "­": "",
})
_WORDISH = re.compile(r"[A-Za-z0-9]")
_NOISE_TOKEN = re.compile(r"^[^\w$%@&#+]+$")          # sadece sembol: | ~ ` ' . , _ vb.
_EDGE_JUNK_L = re.compile(r"^[|_~`'\"\s]+")
_EDGE_JUNK_R = re.compile(r"[|_~`\s]+$")
_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
_URL = re.compile(r"\b(?:https?://|www\.)\S+", re.I)
_MONEY = re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?")
_NUM = re.compile(r"\d+")


def alnum_ratio(s):
    s = s.replace(" ", "")
    return sum(c.isalnum() for c in s) / len(s) if s else 0.0


def clean_text(raw, min_line_alnum=0.4):
    """OCR metnini temizler; büyük/küçük harf, e-posta, tutar gibi bilgiler korunur
    (bilgi çıkarımı aşamasında gerekecek)."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).translate(_CHAR_MAP)
    t = "".join(c for c in t if c == "\n" or c.isprintable())
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)               # satır sonu tirelemesini birleştir
    lines = []
    for line in t.split("\n"):
        tokens = [tok for tok in line.split() if not _NOISE_TOKEN.match(tok)]
        line = " ".join(tokens)
        line = _EDGE_JUNK_R.sub("", _EDGE_JUNK_L.sub("", line))
        if len(line) < 2 or not _WORDISH.search(line):
            continue
        if alnum_ratio(line) < min_line_alnum:              # çoğu sembol olan çöp satır
            continue
        lines.append(line)
    return "\n".join(lines)


def normalize_for_model(clean):
    """Sınıflandırıcı için: küçük harf + e-posta/URL/tutar/sayıları ortak simgeye çevir.
    Böylece model 'john@acme.com' yerine 'bir e-posta adresi var' bilgisini öğrenir."""
    t = clean.lower()
    t = _EMAIL.sub(" emailaddr ", t)
    t = _URL.sub(" urladdr ", t)
    t = _MONEY.sub(" moneyamt ", t)
    t = _NUM.sub("0", t)
    t = re.sub(r"[^a-z0\s]", " ", t)
    return " ".join(t.split())


def garbage_ratio(raw):
    """Ham OCR çıktısındaki anlamsız token oranı (harf/rakam içermeyen ya da
    harf-rakam-sembol karışık kısa parçalar)."""
    toks = raw.split() if isinstance(raw, str) else []
    if not toks:
        return 1.0
    bad = 0
    for tok in toks:
        if not _WORDISH.search(tok):
            bad += 1
        elif len(tok) <= 3 and alnum_ratio(tok) < 0.67:
            bad += 1
    return bad / len(toks)


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/final_model.py
"""Son model: karakter TF-IDF + tablo özellikleri → olasılık veren Linear SVM (Platt kalibrasyonu).

04'teki ek deneyde tablo özelliği (asistan önerisi) Linear SVM'in test başarısını artırdı. Bu modül o
deneyi demoda kullanılabilecek tek bir modele dönüştürür:

    metin (normalize_for_model çıktısı) ──► karakter TF-IDF ─┐
                                                              ├─► Linear SVM (kalibre) ─► olasılıklar
    görüntü ──► detect_table ──► 5 tablo özelliği (ölçekli) ─┘

"Emin değilim": en yüksek olasılık `threshold`'un altındaysa model karar vermez; en olası iki türü
gösterir. Eşik validation setinde seçilir (test setine bakılmaz).
"""
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

TAB_COLS = ["n_hlines", "n_vlines", "n_intersections", "line_ratio", "has_table"]
UNSURE = "emin_degilim"


def char_tfidf():
    # 04'teki karakter TF-IDF ile aynı ayarlar
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9,
                           sublinear_tf=True, max_features=100_000)


def tab_array(rows):
    """detect_table sözlükleri ya da TAB_COLS sütunlu DataFrame → (n, 5) dizi; sayılar log ölçekte."""
    if hasattr(rows, "loc"):
        t = rows[TAB_COLS].astype(float).values.copy()
    else:
        t = np.array([[float(r[c]) for c in TAB_COLS] for r in rows])
    t[:, :3] = np.log1p(t[:, :3])
    return t


class TextTableModel:
    def __init__(self, C=0.5, table_weight=0.2, threshold=0.0, seed=42):
        self.C, self.table_weight, self.threshold, self.seed = C, table_weight, threshold, seed

    def _X(self, texts, tabs, fit=False):
        if fit:
            self.vec_ = char_tfidf().fit(texts)
            self.scaler_ = StandardScaler().fit(tab_array(tabs))
        X_text = self.vec_.transform(texts)
        if self.table_weight == 0:
            return X_text.tocsr()
        X_tab = csr_matrix(self.table_weight * self.scaler_.transform(tab_array(tabs)))
        return hstack([X_text, X_tab]).tocsr()

    def fit(self, texts, tabs, y, calibrate=True):
        X = self._X(texts, tabs, fit=True)
        svm = LinearSVC(C=self.C, random_state=self.seed)
        self.clf_ = CalibratedClassifierCV(svm, cv=5, method="sigmoid").fit(X, y) if calibrate else svm.fit(X, y)
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, texts, tabs):
        return self.clf_.predict_proba(self._X(texts, tabs))

    def predict(self, texts, tabs):
        return self.clf_.predict(self._X(texts, tabs))

    def decide(self, texts, tabs, threshold=None):
        """Tahmin + 'emin değilim': en yüksek olasılık eşiğin altındaysa UNSURE döner."""
        t = self.threshold if threshold is None else threshold
        P = self.predict_proba(texts, tabs)
        top = P.argmax(axis=1)
        labels = self.classes_[top].astype(object)
        labels[P.max(axis=1) < t] = UNSURE
        return labels, P


def coverage_table(y_true, P, classes, thresholds):
    """Her eşik için: kapsama (karar verilen belge oranı), karar verilenlerdeki doğruluk, kaçan hata sayısı."""
    y_true = np.asarray(y_true)
    pred = np.asarray(classes)[P.argmax(axis=1)]
    conf = P.max(axis=1)
    rows = []
    for t in thresholds:
        keep = conf >= t
        n_err_all = int((pred != y_true).sum())
        n_err_kept = int((pred[keep] != y_true[keep]).sum())
        rows.append({
            "esik": t,
            "kapsama": keep.mean(),
            "karar_verilenlerde_dogruluk": (pred[keep] == y_true[keep]).mean() if keep.any() else np.nan,
            "emin_degilim_sayisi": int((~keep).sum()),
            "yakalanan_hata": n_err_all - n_err_kept,
            "kalan_hata": n_err_kept,
            "gereksiz_kararsizlik": int(((~keep) & (pred == y_true)).sum()),
        })
    return rows


In [ ]:
sys.path.insert(0, str(ROOT))
import ocr_core, ocr_batch, text_clean, final_model
for m in (ocr_core, ocr_batch, text_clean, final_model): importlib.reload(m)
from text_clean import clean_text, normalize_for_model

df = pd.read_csv(DATA / "cleaned_dataset.csv")
df = df[df["label"].isin(CLASSES)].reset_index(drop=True)
print(df.groupby(["split", "label"]).size().unstack())

## A. Yeniden OCR (2 kat büyütme)

İlerleme `bitirme_ocr/ocr_x2_ilerleme.txt` dosyasına da yazılır.

In [ ]:
LOG = ROOT / "ocr_x2_ilerleme.txt"
def log(msg):
    print(msg, flush=True)
    with open(LOG, "a") as f:
        f.write(msg + "\n")

OUT = DATA / "ocr_dataset_x2.csv"
x2 = ocr_batch.run_batch(list(df["path"]), DATA, OUT, scale=2.0, workers=max(1, os.cpu_count()), chunk=40, log=log)
print("Bitti:", len(x2), "belge · hata:", int((x2["hata"].fillna("") != "").sum()))

## B. OCR kalitesi: ham ve x2

In [ ]:
raw = pd.read_csv(DATA / "ocr_dataset.csv")
if (DATA / "ocr_dataset_aday.csv").exists():
    raw = pd.concat([raw, pd.read_csv(DATA / "ocr_dataset_aday.csv")], ignore_index=True)
raw = raw.drop_duplicates("path").set_index("path")
x2 = pd.read_csv(DATA / "ocr_dataset_x2.csv").drop_duplicates("path").set_index("path")

from importlib import resources
VOCAB = {l.split()[0] for l in (resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt").read_text().splitlines() if l.strip()}
def dict_ratio(t):
    w = [x.lower() for x in pd.Series(str(t)).str.findall(r"[A-Za-z]{3,}")[0]]
    return sum(x in VOCAB for x in w) / len(w) if w else 0.0

rows = []
for p, lab in zip(df.path, df.label):
    r, n = raw.loc[p], x2.loc[p]
    rows.append({"path": p, "label": lab,
                 "guven_ham": r.get("mean_conf", np.nan), "guven_x2": n["mean_conf"],
                 "kelime_ham": r.get("n_words", np.nan), "kelime_x2": n["n_words"],
                 "sozluk_ham": dict_ratio(r["text"]), "sozluk_x2": dict_ratio(n["text"])})
q = pd.DataFrame(rows)
summ = q.groupby("label")[["guven_ham", "guven_x2", "kelime_ham", "kelime_x2", "sozluk_ham", "sozluk_x2"]].mean()
summ.loc["TÜMÜ"] = q[summ.columns].mean()
summ = summ.rename(index=TR).round(3)
print(summ.to_string())
summ.to_csv(ROOT / "ocr_x2_kalite.csv")
print("\nSözlük oranı artan belge oranı: %{:.1f}".format(100 * (q.sozluk_x2 > q.sozluk_ham).mean()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
s = summ.drop("TÜMÜ")
x = np.arange(len(s))
for k, (a, b, t) in enumerate([("guven_ham", "guven_x2", "Ortalama OCR güveni"), ("sozluk_ham", "sozluk_x2", "Sözlükte bulunan kelime oranı")]):
    ax[k].bar(x - 0.2, s[a], 0.4, label="ham", color="#9aa5b1")
    ax[k].bar(x + 0.2, s[b], 0.4, label="2 kat büyütme", color="#2f6fde")
    ax[k].set_xticks(x, s.index); ax[k].set_title(t); ax[k].legend(); ax[k].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "ocr_x2_kalite.png", dpi=150, bbox_inches="tight"); plt.show()

## C. Sınıflandırma: aynı modeller, yeni metin

Modeller 04 ve 10'da seçilen **aynı ayarlarla** eğitilir; tek fark girdi metnidir. Hangi OCR'ın kullanılacağına **validation macro-F1** ile karar verilir; test seti sadece raporlamak içindir.

In [ ]:
from sklearn.base import clone
from final_model import TextTableModel, TAB_COLS

df["text_x2"] = df["path"].map(x2["text"]).fillna("")
df["norm_x2"] = df["text_x2"].map(clean_text).map(normalize_for_model)
tab = pd.read_csv(DATA / "table_features.csv")
d = df.merge(tab[["path"] + TAB_COLS], on="path", how="left")
d["text_norm"] = d["text_norm"].fillna("")
empty_x2 = d["norm_x2"].str.split().str.len().fillna(0) < 5
train = d[(d.split == "train") & (~d.is_empty) & (~empty_x2)]
val, test = d[d.split == "validation"], d[d.split == "test"]

old = joblib_model = __import__("joblib").load(MODELS / "best_model.joblib")       # 04: kelime TF-IDF + LR
fi = json.loads((MODELS / "final_model.json").read_text())                          # 10: karakter + tablo
res, preds = [], {}
for ocr_name, col in [("ham", "text_norm"), ("x2", "norm_x2")]:
    m04 = clone(old).fit(train[col], train.label)
    mfin = TextTableModel(C=fi["C"], table_weight=fi["tablo_agirligi"], seed=SEED).fit(train[col], train, train.label, calibrate=False)
    for mname, m, pred in [("04 modeli (kelime TF-IDF + LR)", m04, lambda m, f: m.predict(f[col])),
                           ("Son model (karakter + tablo, SVM)", mfin, lambda m, f: m.predict(f[col], f))]:
        pv, pt = pred(m, val), pred(m, test)
        preds[(ocr_name, mname)] = pt
        f1c = f1_score(test.label, pt, labels=CLASSES, average=None)
        res.append({"ocr": ocr_name, "model": mname,
                    "val_macro_f1": f1_score(val.label, pv, average="macro"),
                    "test_accuracy": accuracy_score(test.label, pt), "test_macro_f1": f1_score(test.label, pt, average="macro"),
                    **{f"F1 {TR.get(c, c)}": v for c, v in zip(CLASSES, f1c)},
                    "fatura_diye_yanlis": int(((pt == "invoice") & (test.label.values != "invoice")).sum())})
res = pd.DataFrame(res).round(4)
print(res.to_string(index=False))
res.to_csv(ROOT / "ocr_x2_siniflandirma.csv", index=False)

In [ ]:
for mname in res.model.unique():
    a = preds[("ham", mname)] == test.label.values
    b = preds[("x2", mname)] == test.label.values
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    print(f"{mname}: sadece ham doğru {n01}, sadece x2 doğru {n10}, McNemar p = {p:.4f}")

best = res.sort_values("val_macro_f1", ascending=False).iloc[0]
print(f"\nValidation'a göre en iyi: OCR = {best.ocr}, model = {best.model} (val macro-F1 {best.val_macro_f1:.4f}, test accuracy {best.test_accuracy:.4f})")
(ROOT / "ocr_x2_karar.json").write_text(json.dumps({"ocr": best.ocr, "model": best.model,
    "val_macro_f1": float(best.val_macro_f1), "test_accuracy": float(best.test_accuracy)}, ensure_ascii=False, indent=2))

**Yorum için:** x2 validation'da daha iyiyse, son model (10) ve demo (05) x2 OCR ile yeniden eğitilmelidir; demo zaten küçük yazılı görüntüleri büyüterek okuyor. Fark küçükse ve McNemar p > 0,05 ise, büyütmenin asıl faydası sınıflandırmada değil **içerik çıkarımında** (daha doğru okunan tarih, tutar, ad) görülür.